# Distributed anomaly detection with DMF on Spark

This notebook detects anomalous values in a time series with `dmf.spark`.
It needs the `spark` extra: `pip install "dissimilarity-mapping-filtering[spark]"` and Java 17 or later.

In [1]:
import numpy as np
from pyspark.sql import SparkSession

from dmf.spark import SparkDMF, to_signals

spark = SparkSession.builder.master("local[*]").appName("dmf-example").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Generate a sine wave with three anomalous spikes, at positions 10, 20 and 30.
With real data you would read a table instead, e.g. `data = spark.read.table("my_table")`.

In [2]:
values = np.sin(np.arange(50))
values[[10, 20, 30]] = [5, 19, 20]
data = values.reshape(-1, 1)  # one row per time step, one column

`to_signals` converts the table into an RDD of `(row_index, values)`, casting values to numbers and filling missing ones.

In [3]:
signals = to_signals(data, kind="numerical")
signals.take(3)

[(0, [0.0]), (1, [0.8414709848078965]), (2, [0.9092974268256817])]

Compare every pair of rows (Euclidean distance), add up the distances of each row, and keep the rows whose z-score is above `alpha`.

In [4]:
model = SparkDMF(dissimilarity="euclidean", filtering="zscore", alpha=0)
anomalies = model.detect(signals)
sorted(anomalies.collect())

[(10, 263.2176404615494), (20, 907.2176404615494), (30, 955.2176404615494)]

In [5]:
spark.stop()